In [ ]:
# =============================================================================
# TUTOR FEEDBACK IMPLEMENTATION
# Figures 6, 7, 8 + Table 1 — all in LEVELS (IP in billions)
# Aligns descriptive section with regression analysis, per Franziska's feedback
# =============================================================================

# %% CELL 1 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded.")

# %% CELL 2 — Upload Dataset
df = pd.read_csv('../data/df_merged.csv')
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# %% CELL 3 — Data Preparation
df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))
df = df.sort_values(['country', 'date']).reset_index(drop=True)

# Scaled variables for regression readability
df['ip_billions']      = df['ind_prod_const_sa'] / 1e9
df['us_imp_thousands'] = df['usa_imp_sa'] / 1e3

country_order  = ['Malaysia', 'Philippines', 'Singapore', 'Thailand']
country_colors = {'Malaysia': '#2980b9', 'Philippines': '#4ecdc4',
                  'Singapore': '#1f4e79', 'Thailand': '#f28585'}
print(f"✅ Data ready: {df.shape[0]} rows")

# %% CELL 4 — FIGURE 6 (new): Distribution of IP LEVELS by country (box plots)
# Replaces old YoY box plot — aligns with regression on levels
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8fafc')

data = [df.loc[df['country'] == c, 'ip_billions'].values for c in country_order]
bp = ax.boxplot(data, tick_labels=country_order, patch_artist=True,
                widths=0.55, medianprops={'color': '#e67e22', 'linewidth': 2},
                flierprops={'marker': 'o', 'markerfacecolor': 'white',
                            'markersize': 5, 'markeredgecolor': '#7f8c8d'})
for patch, c in zip(bp['boxes'], country_order):
    patch.set_facecolor(country_colors[c])
    patch.set_edgecolor('white')
    patch.set_linewidth(1.5)

ax.set_title('Figure 6: Distribution of Industrial Production (Levels) by Country',
             fontsize=13, fontweight='bold', color='#1f4e79', pad=12)
ax.set_ylabel('IP (billions, constant, seasonally adjusted)',
              fontsize=11, fontweight='bold')
ax.grid(True, axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig6_levels_boxplot.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Saved fig6_levels_boxplot.png")

# %% CELL 5 — FIGURE 7 (new): IP LEVEL distributions (histograms + KDE)
# Replaces old YoY histograms — aligns with regression on levels
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), dpi=150, sharey=False)
fig.patch.set_facecolor('white')

for ax, c in zip(axes, country_order):
    x = df.loc[df['country'] == c, 'ip_billions']
    col = country_colors[c]

    # Histogram
    n_bins = 25
    ax.hist(x, bins=n_bins, color=col, edgecolor='white', alpha=0.75)

    # KDE overlay (scaled to histogram counts)
    kde = sps.gaussian_kde(x)
    xs = np.linspace(x.min(), x.max(), 200)
    scale = len(x) * (x.max() - x.min()) / n_bins
    ax.plot(xs, kde(xs) * scale, color=col, linewidth=2.2)

    # Mean / median reference lines
    m, md = x.mean(), x.median()
    ax.axvline(m,  color='#e74c3c', linestyle='--', linewidth=1.5,
               label=f'Mean = {m:.1f} B')
    ax.axvline(md, color='#f39c12', linestyle=':',  linewidth=1.8,
               label=f'Median = {md:.1f} B')

    ax.set_title(c, fontsize=12, fontweight='bold', color=col)
    ax.set_xlabel('IP (billions)', fontsize=10)
    ax.set_ylabel('Count' if ax is axes[0] else '', fontsize=10)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Figure 7: IP Level Distributions by Country (Histogram + KDE)',
             fontsize=13, fontweight='bold', color='#1f4e79', y=1.02)
plt.tight_layout()
plt.savefig('fig7_levels_hist.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Saved fig7_levels_hist.png")

# %% CELL 6 — FIGURE 8 (new): Country-level scatter — US imports vs IP (levels)
# Replaces old YoY scatter — directly visualises Equation 1 on levels
fig, axes = plt.subplots(2, 2, figsize=(13, 10), dpi=150)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

# Also collect Table 1 data during this loop
table1_rows = []

for ax, c in zip(axes_flat, country_order):
    sub = df[df['country'] == c]
    x = sub['us_imp_thousands'].values
    y = sub['ip_billions'].values
    col = country_colors[c]

    # Scatter
    ax.scatter(x, y, s=20, color=col, alpha=0.55, edgecolor='none')

    # OLS fit + 95% CI band
    X = sm.add_constant(x)
    ols = sm.OLS(y, X).fit()
    beta = ols.params[1]
    r2   = ols.rsquared
    r    = np.sqrt(r2) * np.sign(beta)
    pval = ols.pvalues[1]

    xs = np.linspace(x.min(), x.max(), 200)
    Xs = sm.add_constant(xs)
    pred = ols.get_prediction(Xs).summary_frame(alpha=0.05)
    ax.plot(xs, pred['mean'], color='#e74c3c', linewidth=2.2)
    ax.fill_between(xs, pred['mean_ci_lower'], pred['mean_ci_upper'],
                    color='#e74c3c', alpha=0.15)

    # Inset stats (upper-left)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 \
          else '*' if pval < 0.05 else 'ns'
    stats_text = f'r = {r:.3f}  {sig}\nβ = {beta:.4f}\nR² = {r2:.3f}'
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#bdc3c7', linewidth=1.0))

    # Trade dep inset (lower-right)
    td = sub['exports_gdp_pct'].mean()
    ax.text(0.97, 0.03, f'Trade Dep: {td:.0f}%',
            transform=ax.transAxes, fontsize=9, style='italic',
            color='#7f8c8d', va='bottom', ha='right')

    ax.set_title(c, fontsize=13, fontweight='bold', color=col)
    ax.set_xlabel('US Imports (thousands)', fontsize=11)
    ax.set_ylabel('Industrial Production (billions)', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Collect Table 1 row
    table1_rows.append({
        'Country': c, 'beta': beta, 'R_squared': r2,
        'r_pearson': r, 'p_value': pval, 'N': len(sub),
        'Trade_Dep_pct': td,
    })

fig.suptitle('Figure 8: Country-Level OLS — US Imports vs Industrial Production (Levels)',
             fontsize=14, fontweight='bold', color='#1f4e79', y=1.00)
plt.tight_layout()
plt.savefig('fig8_levels_scatter.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Saved fig8_levels_scatter.png")

# %% CELL 7 — TABLE 1 (new): Country-Level OLS Results (levels)
# Replaces YoY-based Table 1 — aligns with Fig 8 and Table 2
table1 = pd.DataFrame(table1_rows)
table1_display = table1.copy()
table1_display['beta']         = table1_display['beta'].round(4)
table1_display['R_squared']    = table1_display['R_squared'].round(3)
table1_display['r_pearson']    = table1_display['r_pearson'].round(3)
table1_display['p_value']      = table1_display['p_value'].apply(
    lambda p: f"< 0.001 ***" if p < 0.001 else f"{p:.4f}"
)
table1_display['Trade_Dep_pct'] = table1_display['Trade_Dep_pct'].round(1)

# Sort by beta descending (matches report ordering)
table1_display = table1_display.sort_values('beta', ascending=False).reset_index(drop=True)

print("\n" + "=" * 78)
print("TABLE 1 — Country-Level OLS Results (Equation 1)")
print("DV: IP (billions)  |  Regressor: US Imports (thousands)  |  n = 228 per country")
print("=" * 78)
print(table1_display.to_string(index=False))
print("=" * 78)
print("Note: *** p < 0.001. Full statsmodels summaries in Appendix A.")

# Save as CSV for easy import into Word
table1_display.to_csv('table1_country_ols_levels.csv', index=False)
print("\n✅ Saved table1_country_ols_levels.csv")

# %% CELL 8 — Full statsmodels summaries (for Appendix A — transparency)
print("\n" + "=" * 78)
print("FULL STATSMODELS OUTPUT — per country (for Appendix A)")
print("=" * 78)
for c in country_order:
    sub = df[df['country'] == c]
    X = sm.add_constant(sub['us_imp_thousands'])
    y = sub['ip_billions']
    ols = sm.OLS(y, X).fit()
    print(f"\n### {c.upper()} ###")
    print(ols.summary())